In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/cleaned_retail.csv")

df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [2]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].dtype


dtype('<M8[ns]')

In [3]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
snapshot_date


Timestamp('2011-12-10 12:50:00')

In [4]:
rfm = (
    df.groupby("CustomerID")
      .agg({
          "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
          "InvoiceNo": "nunique",
          "TotalPrice": "sum"
      })
      .reset_index()
)

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [5]:
rfm.describe()


,CustomerID,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,92.536422,4.272015,2054.266460
std,1721.808492,100.014169,7.697998,8989.230441
min,12346.000000,1.000000,1.000000,3.750000
25%,13813.250000,18.000000,1.000000,307.415000
50%,15299.500000,51.000000,2.000000,674.485000
75%,16778.750000,142.000000,5.000000,1661.740000
max,18287.000000,374.000000,209.000000,280206.020000


### RFM Distribution Check

RFM metrics show skewed distributions, particularly for Frequency and Monetary.
Quantile-based scoring is therefore used to create balanced customer segments.


In [6]:
rfm["R_score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1])

rfm["F_score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

rfm["M_score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4])


In [7]:
rfm["RFM_Score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

rfm.head()


,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,12346,326,1,77183.60,1,1,4,114
1,12347,2,7,4310.00,4,4,4,444
2,12348,75,4,1797.24,2,3,4,234
3,12349,19,1,1757.55,3,1,4,314
4,12350,310,1,334.40,1,1,2,112


In [8]:
def segment_customers(row):
    if row["R_score"] >= 3 and row["F_score"] >= 3 and row["M_score"] >= 3:
        return "Champions"
    elif row["R_score"] >= 3 and row["F_score"] >= 2:
        return "Loyal Customers"
    elif row["R_score"] >= 3 and row["F_score"] <= 2:
        return "Potential Loyalists"
    elif row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    elif row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Lost Customers"
    else:
        return "Others"

rfm["Segment"] = rfm.apply(segment_customers, axis=1)
rfm["Segment"].value_counts()


Segment
Lost Customers         1504
Champions              1319
At Risk                 646
Loyal Customers         610
Potential Loyalists     259
Name: count, dtype: int64

### Customer Segmentation Logic

Customers are grouped into business-relevant segments using RFM score
combinations. This enables targeted retention, reactivation, and loyalty
strategies instead of uniform marketing actions.


### Segment-wise Business Insights
- Champions: High-value and highly engaged customers.
  Strategy: Reward loyalty, early access, premium offers.

- Loyal Customers: Frequent purchasers with moderate spend.
  Strategy: Upsell and cross-sell to increase monetary value.

- Potential Loyalists: Recent but infrequent buyers.
  Strategy: Personalized follow-ups and onboarding campaigns.

- At Risk: Previously active customers with declining engagement.
  Strategy: Re-engagement campaigns, limited-time discounts.

- Lost Customers: Inactive and low-engagement customers.
  Strategy: Low-cost reactivation or exclude from paid marketing.


In [9]:
rfm.to_csv("../data/processed/rfm_segments.csv", index=False)
